In [2]:
import pandas as pd
import numpy as np 
import seaborn as sns 
import plotly.express as px 
from pathlib import Path
import geopandas as gps

In [ ]:
df = pd.read_excel("dados.xlsx", sheet_name = "Dados")

df

In [ ]:
teste = df.groupby(["data","setor","turno"], as_index = False).aggregate(peso = ("producao", "sum"), dia_semana = ("dia_semana", "unique"))
teste["dia_semana"] = [i[0] for i in teste["dia_semana"]]
teste["data"] = pd.to_datetime(teste["data"], dayfirst= True)
teste = teste[(teste["data"]>="20/05/2025") & (teste["data"]<="20/06/2025") & (teste["dia_semana"].isin(["seg","qua","sex"]))]
px.line(teste, x = "data", y = "peso", color = "setor")

In [4]:
def serie_historica(dados, data_ini, data_fin, dia_sem ,turn ,var):
    teste = dados
    teste["data"] = pd.to_datetime(teste["data"], dayfirst= True)
    teste = dados.groupby(["data", "setor", "turno"], as_index = False).aggregate(var = (var,"sum"), dia_semana = ("dia_semana", "unique"))
    teste["dia_semana"] = [i[0] for i in teste["dia_semana"]]
    teste = teste[(teste["data"]>=data_ini) & (teste["data"]<=data_fin) & (teste["turno"].isin(turn)) & (teste["dia_semana"].isin(dia_sem))]
    
    teste.sort_values(by = "data", inplace = True)
    teste["indexes"] = teste["data"].dt.strftime("%d/%m/%y") + "<br>" + teste["dia_semana"]
    
    figura = px.line(teste, x = "indexes", y = "var", color = "setor")
    figura.show()
    
    return teste

In [ ]:
treino  = serie_historica(df, "20/05/2025", "20/10/2025", ["seg","qua","sex"], ["DIURNO"] ,"horas_coleta")

In [6]:
def medias(dados, data_ini, data_fin, dia_sem ,turn ,var):
    teste = dados
    teste["data"] = pd.to_datetime(teste["data"], dayfirst= True)
    
    teste = teste[(teste["data"] >= data_ini) & (teste["data"] <= data_fin) & (teste["turno"].isin(turn)) & (teste["dia_semana"].isin(dia_sem))].copy()
    
    teste = teste.groupby(["setor", "dia_semana" ,"turno"], as_index= False)[var].mean()
    
    figura = px.bar(teste, x = var, y = "setor", color = "dia_semana", orientation= "h")
    figura.update_layout(barmode = "group")
    figura.show()
    

In [ ]:
medias(df, "20/05/2025", "20/06/2025",["seg","qua","sex"], ["DIURNO"], "producao")

In [ ]:
!pip install geopandas

In [103]:

mapa_atual = gps.read_file("Atual/01.shp")

for i in range(2,22):
    if i <10: 
        mapa = gps.read_file(f"Atual/0{i}.shp")
        mapa_atual = pd.concat([mapa_atual, mapa])
    else:
        mapa = gps.read_file(f"Atual/{i}.shp")
        mapa_atual = pd.concat([mapa_atual, mapa])

In [105]:
mapa_atual = mapa_atual.to_crs(epsg=4326)

mapa_atual.to_file("mapa_atual.geojson", driver = "GeoJSON")

In [8]:
mapa_atual = gps.read_file("mapa_atual.geojson")
df = pd.read_excel("dados.xlsx",sheet_name = "Dados")
teste = df.groupby( "setor", as_index = False)["producao"].mean()
teste.drop([21, 22], axis = 0, inplace = True)
teste["setor"] = [f"0{i}" if i < 10 else f"{i}" for i in teste["setor"]]


In [12]:
mapa

{'type': 'FeatureCollection',
 'features': [{'id': '0',
   'type': 'Feature',
   'properties': {'IDENTIFICA': '00101',
    'SETOR': '01',
    'APELIDO': '01',
    'MACROSETOR': 'RIO CLARO - DOMICILIAR',
    'COR': 3328050.0,
    'FREQUENCIA': 'TER/QUI/SAB - NOTURNO'},
   'geometry': {'type': 'Polygon',
    'coordinates': (((-47.563047, -22.395606),
      (-47.563029, -22.395806),
      (-47.563021, -22.396073),
      (-47.563149, -22.39713),
      (-47.563353, -22.398674),
      (-47.563492, -22.399212),
      (-47.563735, -22.399632),
      (-47.563744, -22.399647),
      (-47.563806, -22.399753),
      (-47.563875, -22.399873),
      (-47.56396, -22.400022),
      (-47.564137, -22.400306),
      (-47.564212, -22.40046),
      (-47.564252, -22.400541),
      (-47.564352, -22.400748),
      (-47.564448, -22.400946),
      (-47.564535, -22.401124),
      (-47.564661, -22.401418),
      (-47.564801, -22.401746),
      (-47.564903, -22.401984),
      (-47.564987, -22.40238),
      (-47.56

In [11]:
mapa = mapa_atual.__geo_interface__

In [ ]:
fig = px.choropleth(teste, geojson=mapa, locations="setor", featureidkey="properties.SETOR", color="producao", color_continuous_scale="Oranges")
fig.update_layout(width=500,height=700, margin = dict(l=4, r=0, t=0, b=0), coloraxis_colorbar=dict(len=0.6))
fig.update_geos(projection_type="mercator", fitbounds = "geojson", visible = False, domain=dict(x=[0, 1], y=[0, 1]), projection_scale = 1 )
fig.show()

In [ ]:
fig = px.choropleth_map(
    teste,
    geojson=mapa,
    locations="setor",
    featureidkey="properties.SETOR",
    color="producao",
    color_continuous_scale="Oranges",
    center = {"lat": -22.39, "lon": -47.577}, 
    zoom = 11)

fig.update_layout(
    margin=dict(l=8, r=0, t=0, b=0), 
    width = 500, 
    height = 900)

fig.update_traces(marker=dict(opacity=0.5))

In [102]:
def mapa_analitico(variavel, data_ini, data_fin, turno, dia_semana):
    
    #importação dos dados
    dados = pd.read_excel("dados.xlsx", sheet_name= "Dados")
    dados["data"] = pd.to_datetime(dados["data"], dayfirst= True)
    mapa = gps.read_file("mapa_atual.geojson")
    
    
    #aplicação dos filtros 
    dados_analise = dados[(dados["data"]>= data_ini) &
                          (dados["data"]<= data_fin) &
                          (dados["turno"].isin(turno)) &
                          (dados["dia_semana"].isin(dia_semana))]
    
    #calculo das médias da variável selecionada para cada setor 
    medias = dados_analise.groupby("setor", as_index = False)[variavel].mean()
    medias["setor"] = [f"0{i}" if i <10 else f"{i}" for i in medias["setor"]]
    
    #aplicação do filtro no arquivo shp 
    mapa_filtro = mapa[mapa["SETOR"].isin(medias["setor"])]
    mapa_plot = mapa_filtro.__geo_interface__
    
    # return mapa_filtro, medias - linha de teste 
    # plot do gráfico georeferenciado 
    fig = px.choropleth_map(
        medias,
        geojson=mapa_plot,
        locations="setor",
        featureidkey="properties.SETOR",
        color= variavel,
        color_continuous_scale="Oranges",
        center = {"lat": -22.39, "lon": -47.577}, 
        zoom = 11, 
        opacity = 0.5)
    
    fig.update_layout(
        margin=dict(l=8, r=0, t=0, b=0), 
        width = 500, 
        height = 900)
    
    fig.show()
    

In [ ]:
mapa_analitico("producao", "20/05/2025", "20/06/2025", ["DIURNO"], ["seg","qua"])

In [106]:
df[df["dia_semana"]=="sáb"]

,data,mês,dia_mes,dia_semana,setor,turno,carro,tempo_coleta,horas_coleta,tempo_trajeto,horas_trajeto,tempo_aterro,horas_aterro,km_coleta,km_trajeto,producao,producao_ton,viagens,ton/h,ton/km
47,2025-04-05,abr,5,sáb,8,DIURNO,31912,07:34:00,7.566667,01:50:00,1.833333,00:20:00,0.333333,34,46,14270,14.27,2,1.885903,0.419706
48,2025-04-05,abr,5,sáb,10,DIURNO,33049,04:23:00,4.383333,00:27:00,0.450000,00:25:00,0.416667,36,16,12190,12.19,1,2.780989,0.338611
49,2025-04-05,abr,5,sáb,12,DIURNO,32049,07:30:00,7.500000,01:23:00,1.383333,00:45:00,0.750000,37,51,15350,15.35,2,2.046667,0.414865
50,2025-04-05,abr,5,sáb,15,DIURNO,32048,05:00:00,5.000000,01:00:00,1.000000,00:15:00,0.250000,40,19,13990,13.99,2,2.798000,0.349750
51,2025-04-05,abr,5,sáb,17,DIURNO,33049,05:50:00,5.833333,00:50:00,0.833333,00:15:00,0.250000,48,15,16010,16.01,2,2.744571,0.333542
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2905,2026-01-03,jan,3,sáb,6,NOTURNO,33049,03:52:00,3.866667,00:54:00,0.900000,00:12:00,0.200000,21,21,11560,11.56,1,2.989655,0.550476
2906,2026-01-03,jan,3,sáb,11,NOTURNO,32051,02:50:00,2.833333,00:25:00,0.416667,00:15:00,0.250000,12,12,9030,9.03,1,3.187059,0.752500
2907,2026-01-03,jan,3,sáb,11,NOTURNO,32049,02:45:00,2.750000,00:19:00,0.316667,00:17:00,0.283333,11,12,9280,9.28,1,3.374545,0.843636
2908,2026-01-03,jan,3,sáb,11,NOTURNO,32051,03:05:00,3.083333,00:40:00,0.666667,00:25:00,0.416667,15,12,8460,8.46,1,2.743784,0.564000
